# Milestone 2 lab, notebook 2 of 8: Remember where you stopped

## Where we are

| Notebook | What you do |
|---|---|
| 1. Ask only for what changed | Read the orders that changed, not all of them |
| **2. Remember where you stopped** (you are here) | Keep a watermark. See what happens when it moves too early |
| 3. When the API says no | Meet the rate limit and a server error. Wait, and try again |
| 4. Land it raw | Put every order in Postgres, exactly as it arrived |
| 5. Pull or push | A short lab: let the order system send the changes to you |
| 6. What you built has a name | Change data capture, and what this kind cannot see |
| 7. Counted twice | Run a few mornings. Watch the report go wrong |
| 8. The production version | The same logic as a tested program |

In notebook 1 you typed the time of yesterday's run by hand. A pipeline cannot do that. It must **remember how far it got**.

In [1]:
import json

from lab_helpers import LAB_DIR, api, read_orders, set_morning, show_doc

## Step 1: What exactly should it remember?

The easy idea is "the time my job ran". Look at Monday's run, and at the newest change it actually read.

In [2]:
set_morning("2026-06-08")
monday, _ = read_orders()

job_ran_at = api.get("/admin/clock").json()["now_utc"]
newest_read = max(order["updated_at"] for order in monday)

print("The job ran at:              ", job_ran_at)
print("The newest change it read at:", newest_read)

It is now Monday 2026-06-08 07:00:00 in Arden.
The job ran at:               2026-06-08T12:00:00Z
The newest change it read at: 2026-06-08T03:48:53Z


These are two different times, from two different clocks. One clock belongs to your job. The other belongs to the order system, which wrote `updated_at`.

Two computers never agree exactly on the time. If you save your own clock and it is a few seconds ahead, a change in those seconds is skipped for ever. So the rule is:

> Save the largest `updated_at` you read, not the time the job ran.

That saved value is the **watermark**.

## Step 2: Keep it somewhere

For now, a small file. Notebook 4 moves it into the warehouse.

In [3]:
STATE = LAB_DIR / "lab_watermark.json"

def save_watermark(value):
    STATE.write_text(json.dumps({"orders": value}))

def read_watermark():
    return json.loads(STATE.read_text())["orders"] if STATE.exists() else None

save_watermark(newest_read)
print("Saved watermark:", read_watermark())

Saved watermark: 2026-06-08T03:48:53Z


## Step 3: The next morning uses it

Nobody types a time any more.

In [4]:
set_morning("2026-06-09")

watermark = read_watermark()
changed, calls = read_orders(updated_since=watermark)
print(f"Asked for orders changed since {watermark}")
print(f"Got {len(changed)} orders in {calls} calls.")

save_watermark(max(order["updated_at"] for order in changed))
print("New watermark:", read_watermark())

It is now Tuesday 2026-06-09 07:00:00 in Arden.
Asked for orders changed since 2026-06-08T03:48:53Z
Got 29 orders in 2 calls.
New watermark: 2026-06-09T04:17:31Z


## Step 4: What happens at the boundary?

Ask again at once, with the new watermark. Nothing has changed since, so you might expect nothing back.

In [5]:
again, _ = read_orders(updated_since=read_watermark())

for order in again:
    print(f"order {order['order_id']}, updated_at {order['updated_at']}")

order 1251, updated_at 2026-06-09T04:17:31Z


One order comes back: the one whose `updated_at` **equals** the watermark. The API's filter is "at or after", not "after".

That looks like a waste. It is a choice. Two orders can change in the same second. If the filter were "after", the second of them could be lost when it is saved a moment later. So the choice is between two mistakes:

| Filter | At the boundary | The mistake it can make |
|---|---|---|
| after (`>`) | The boundary order is skipped | A row is **lost**. Nothing shows it |
| at or after (`>=`) | The boundary order is read again | A row is **repeated** |

A repeat can be cleaned up later. A lost row cannot. Remember this choice. It comes back in notebook 7.

## Step 5: Move the watermark too early, and lose a morning

When should the watermark be saved: before the rows are written, or after? Try the wrong order, with a crash in the middle.

In [6]:
set_morning("2026-06-08")
save_watermark(max(o["updated_at"] for o in read_orders()[0]))      # Monday is done
warehouse = set()                                                    # pretend warehouse: order ids

set_morning("2026-06-09")
changed, _ = read_orders(updated_since=read_watermark())

save_watermark(max(o["updated_at"] for o in changed))    # 1. the watermark moves first ...
print("💥 the job crashes here, before it writes the rows")
# warehouse.update(...)                                   # 2. ... and this line never runs

retry, _ = read_orders(updated_since=read_watermark())   # the job is run again
print(f"\nThe second try asks from the new watermark and gets {len(retry)} order.")
print(f"Tuesday's {len(changed)} changed orders are never loaded. No error was shown.")

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
💥 the job crashes here, before it writes the rows

The second try asks from the new watermark and gets 1 order.
Tuesday's 29 changed orders are never loaded. No error was shown.


Now the right order: write the rows first, save the watermark after.

In [7]:
set_morning("2026-06-08")
save_watermark(max(o["updated_at"] for o in read_orders()[0]))
set_morning("2026-06-09")
changed, _ = read_orders(updated_since=read_watermark())

print("💥 the job crashes here, before it writes the rows or moves the watermark")

retry, _ = read_orders(updated_since=read_watermark())   # the job is run again
print(f"\nThe second try asks from the OLD watermark and gets all {len(retry)} orders again.")

It is now Monday 2026-06-08 07:00:00 in Arden.
It is now Tuesday 2026-06-09 07:00:00 in Arden.
💥 the job crashes here, before it writes the rows or moves the watermark

The second try asks from the OLD watermark and gets all 29 orders again.


Nothing is lost. The design document has the full picture.

In [8]:
show_doc("HLD.md", "5. Why the watermark is saved with the data")

**From `docs/HLD.md`**

> ## 5. Why the watermark is saved with the data
> 
> Two ways to get it wrong:
> 
> | Order of steps | If the job crashes between them | Result |
> |---|---|---|
> | Save the watermark, then the rows | The watermark has moved. The rows are not there | ❌ Rows are lost for ever. No error shows |
> | Save the rows, then the watermark | The rows are there. The watermark has not moved | ⚠️ The next run reads them again: duplicates |
> | **Both in one transaction** (chosen) | Neither is saved | ✅ The next run starts from the same place |

In [9]:
STATE.unlink()

## A new word

| Word | Meaning | Here |
|---|---|---|
| **Watermark** | "The saved value that marks how far the last run read." It is the largest value of the incremental key that you have loaded. Also called a high-water mark | `2026-06-08T03:48:53Z` after Monday's run |

The definition is from the D8Loop chapter [Extraction: Pulling Data Reliably](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/extraction-pulling-data-reliably).

## What you now know

- The watermark is the largest `updated_at` you loaded. It is not the time the job ran.
- The filter includes the boundary, so one order is read again. A repeat is safer than a loss.
- Move the watermark only after the rows are written. Moved too early, a crash loses a whole morning with no error.

**Say it in an interview:** "My watermark is the largest `updated_at` I have loaded, never the job's clock. I save it in the same transaction as the rows. If I saved it first and the job crashed, the next run would start after rows that were never written."

Next: [notebook 3](03_when_the_api_says_no.ipynb) makes the load survive an order system that says no.